# Battery Switch

## Schema
> <div style="text-align: center;">
> <img src="./bat_switch_schema.png" alt="N-Type MOSFET Usage" style="display: block; margin: 0 auto; width: 100%;"/>
> </div>

### Initial Conditions

In [1]:
from IPython.display import display, Markdown

VccTolerance = 5  # % The tolerance for the supply voltage, which accounts for variations in the battery
# voltage and other factors. This means that the circuit should be able to operate correctly even if the
# supply voltage is up to 5% higher than the maximum battery voltage.
Vbat_max = 14.8  # V, the maximum battery voltage (4S LiPo)
Vcc = Vbat_max * (
    1 + VccTolerance / 100
)  # V, the maximum supply voltage for the circuit
Ic_q2 = 0.3  # A, the maximum collector current of Q2


# Transistor
name = "BC327"
hfe_min_q2 = 100  # the minimum current gain (hFE) of the transistor, which is a common PNP transistor used in low-power applications.
hfe_max_q2 = (
    600 / 2
)  # the maximum current gain (hFE) of the transistor, which is a common PNP transistor used in low-power applications.

display(
    Markdown(
        f"The maximum battery voltage is $V_{{bat, max}} = {Vbat_max:.2f}\\,\\text{{V}}$ and the tolerance is {VccTolerance}\\%."
    )
)
display(
    Markdown(
        f"The $V_{{CC}} = V_{{bat, max}} + {VccTolerance}\\% = {Vcc:.2f}\\,\\text{{V}}$"
    )
)
display(
    Markdown(
        f"The maximum collector current of the transistor Q2 is $I_{{c, q2}} = {Ic_q2}\\,\\text{{A}}$"
    )
)
display(
    Markdown(
        f"The transistor used is a {name} with a minimum current gain of $h_{{FE, min}} = {hfe_min_q2}$ and a maximum current gain of $h_{{FE, max}} = {hfe_max_q2}$."
    )
)

The maximum battery voltage is $V_{bat, max} = 14.80\,\text{V}$ and the tolerance is 5\%.

The $V_{CC} = V_{bat, max} + 5\% = 15.54\,\text{V}$

The maximum collector current of the transistor Q2 is $I_{c, q2} = 0.3\,\text{A}$

The transistor used is a BC327 with a minimum current gain of $h_{FE, min} = 100$ and a maximum current gain of $h_{FE, max} = 300.0$.

<div style="text-align: center;">
<img src="./q3_equivalent_schema.drawio.svg" alt="Q3 Equivalent schema" style="display: block; margin: 0 auto; width: 30%;"/>
</div>

> The maximum base current of Q2 is equal:
> \begin{equation}
> I_{b,q2\ max} = \frac {I_{c, q3}}{h_{FE,q2}}
> \end{equation}

In [2]:
# Ib_q2 = Ic_q2 / hfe_min_q2 # A, the base current of Q2, which is calculated by dividing the collector current by the minimum current gain of the transistor. This ensures that Q1 can provide enough current to drive Q3.
Ib_q2 = (
    Ic_q2 / hfe_max_q2
)  # A, the base current of Q2, which is calculated by dividing the collector current by the maximum current gain of the transistor.

display(
    Markdown(
        f"The base current of Q2 is $I_{{b, q2\\ max}} = \\frac{{I_{{c, q2}}}}{{h_{{FE, max}}}} = {Ib_q2 * 1e6:.2f}\\,\\text{{uA}}$."
    )
)

The base current of Q2 is $I_{b, q2\ max} = \frac{I_{c, q2}}{h_{FE, max}} = 1000.00\,\text{uA}$.

> Lets calculate the $I_{R4}$ value:
> \begin{equation}
> I_{R4} = I_{c,q2} + I_{b, q2\ max} 
> \end{equation}

In [3]:
Ir4 = Ic_q2 + Ib_q2

display(
    Markdown(
        f"The current through R4 is $I_{{R4}} = I_{{c, q2}} + I_{{b, q2}} = {Ir4 * 1000:.2f}\\,\\text{{mA}}$."
    )
)

The current through R4 is $I_{R4} = I_{c, q2} + I_{b, q2} = 301.00\,\text{mA}$.

> Lets calculate the $V_{R4}$ value:
> \begin{equation}
> V_{R4} = V_{CC} * K_{fb}, \text{, where } K_{fb} \text{ - the feedback factor}
> \end{equation}
>
> And calculate the $R4$
>
> \begin{equation}
> R4 = \frac {V_{R4}}{I_{R4}}
> \end{equation}

In [4]:
Kfb = (
    1 / 10
)  # The feedback factor, which is a design parameter that determines how much of the output voltage is fed back to the input. A common value for Kfb is 10, which means that the output voltage is divided by 10 before being fed back to the input.
V_r4 = Vcc * Kfb
R4 = V_r4 / Ir4

display(
    Markdown(
        f"The voltage across R4 is $V_{{R4}} = V_{{CC}} \\times {Kfb} = {V_r4:.2f}\\,\\text{{V}}$."
    )
)
display(
    Markdown(
        f"The value of R4 is $R_{{4}} = \\frac{{V_{{R4}}}}{{I_{{R4}}}} = {R4:.2f}\\,\\Omega$."
    )
)

The voltage across R4 is $V_{R4} = V_{CC} \times 0.1 = 1.55\,\text{V}$.

The value of R4 is $R_{4} = \frac{V_{R4}}{I_{R4}} = 5.16\,\Omega$.

> Lets calculate the $V_{BE,q2}$ value:
> \begin{equation}
> V_{BE,q2} = ln(\frac {I_{d\ max,\ q2}}{I_{sb, q2}} +1) \cdot n \cdot V_T
> \end{equation}
>
> , where 
>
> \begin{equation}
> V_T = \frac {k \cdot T} {q}, \text{ } k = 1.381 \cdot 10^{-23}, \text{ } q=1.602 \cdot 10^{-19}, \text{ } n \text{ is in range 1.05 - 1.2} 
> \end{equation}

In [5]:
from scipy.constants import Boltzmann, elementary_charge

T = 300  # K
k = Boltzmann  # J/K, Boltzmann constant
q = elementary_charge  # C, elementary charge

V_T = k * T / q

display(
    Markdown(
        f"The thermal voltage at room temperature ({T:.1f} K) is $V_{{T}} = \\frac{{k \\cdot T}}{{q}} = {V_T * 1000:.2f}\\,\\text{{mV}}$."
    )
)

The thermal voltage at room temperature (300.0 K) is $V_{T} = \frac{k \cdot T}{q} = 25.85\,\text{mV}$.

In [6]:
import numpy as np

n = 1  # ideality factor, typically between 1 and 2 for silicon diodes
I_sb_q2 = 1e-10  # A, the reverse saturation current of the base-emitter junction of Q2, which is a very small current that flows when the junction is reverse-biased. This value is typically in the range of picoamperes (pA) to femtoamperes (fA) for silicon transistors.

Vbe_q2 = V_T * n * np.log(Ic_q2 / I_sb_q2)

display(
    Markdown(
        f"The base-emitter voltage of Q2 is $V_{{BE, q2}} = V_{{T}} \\cdot n \\cdot \\ln\\left(\\frac{{I_{{c, q2}}}}{{I_{{sb, q2}}}}\\right) = {Vbe_q2:.2f}\\,\\text{{V}}$."
    )
)

The base-emitter voltage of Q2 is $V_{BE, q2} = V_{T} \cdot n \cdot \ln\left(\frac{I_{c, q2}}{I_{sb, q2}}\right) = 0.56\,\text{V}$.

> Now lets calculate $V_{TH}$:

<div style="text-align: center;">
<img src="./q3_vth_theven.drawio.svg" alt="Q2 Vth" style="display: block; margin: 0 auto; width: 30%;"/>
</div>

> Calculate $R_{th}$ - Theven resistance:
>
> \begin{equation}
> R_{TH} = \frac {R4 \cdot (h_{FE} + 1)}{10}
> \end{equation}

In [7]:
Rth = Vbe_q2 / Ib_q2
display(
    Markdown(
        f"The Thevenin equivalent resistance seen by the base of Q2 is $R_{{th}} = \\frac{{V_{{BE, q2}}}}{{I_{{b, q2}}}} = {Rth:.2f}\\,\\Omega$."
    )
)

The Thevenin equivalent resistance seen by the base of Q2 is $R_{th} = \frac{V_{BE, q2}}{I_{b, q2}} = 564.14\,\Omega$.

> So, according to Kirchhoff’s Voltage Law (KVL):
>
> \begin{equation}
> V_{TH} - V_{R_{th}} - V_{BE, q2} - V_{R4} \implies V_{TH} = V_{R_{th}} + V_{BE, q2} + V_{R4} \text{, where } V_{R_{th}} = R_{TH} \cdot I_{b, q2}
> \end{equation}

In [8]:
V_rth = Rth * Ib_q2
display(
    Markdown(
        f"The voltage drop across the Thevenin equivalent resistance is $V_{{Rth}} = R_{{th}} \\times I_{{b, q2}} = {V_rth:.2f}\\,\\text{{V}}$."
    )
)

V_th = V_rth + Vbe_q2 + V_r4
display(
    Markdown(
        f"The Thevenin equivalent voltage is $V_{{th}} = V_{{Rth}} + V_{{BE, q2}} + V_{{R4}} = {V_th:.2f}\\,\\text{{V}}$."
    )
)

The voltage drop across the Thevenin equivalent resistance is $V_{Rth} = R_{th} \times I_{b, q2} = 0.56\,\text{V}$.

The Thevenin equivalent voltage is $V_{th} = V_{Rth} + V_{BE, q2} + V_{R4} = 2.68\,\text{V}$.

> Now we can calculate the devder $R3$ and $R2$:
> 
> \begin{equation}
> R2 = \frac {R_{TH} \cdot (V_{CC}-V_{CEsat,q1})}{V_{TH}} \text{, where} V_{CEsat,q1} - \text{ - Collector-Emmitter saturation voltage}
> \end{equation}

In [9]:
# Q1 parameters
Vce_sat_q1 = 0.14  # The saturation voltage of Q1, which is the voltage drop across the transistor when it is fully on. This is typically around 0.7V for a bipolar junction transistor (BJT).
display(
    Markdown(
        f"The saturation voltage of Q1 is $V_{{CEsat,q1}}= {Vce_sat_q1}\\,\\text{{V}}$."
    )
)

R2 = Rth * (Vcc - Vce_sat_q1) / V_th
display(
    Markdown(
        f"The value of R2 is $R_{{2}} = \\frac{{R_{{th}} \\cdot (V_{{CC}} - V_{{CEsat,q1}})}}{{V_{{th}}}} = {R2:.2f}\\,\\Omega$."
    )
)

The saturation voltage of Q1 is $V_{CEsat,q1}= 0.14\,\text{V}$.

The value of R2 is $R_{2} = \frac{R_{th} \cdot (V_{CC} - V_{CEsat,q1})}{V_{th}} = 3238.94\,\Omega$.

> The $R3$ is:
> 
> \begin{equation}
> R3 = \frac {R2 \cdot R_{TH}}{R2 + R_{TH}}
> \end{equation}

In [10]:
R3 = (R2 * Rth) / (R2 - Rth)

display(
    Markdown(
        f"The value of R3, which is the parallel combination of R2 and Rth, is $R_{{3}} = \\frac{{R_{{2}} \\cdot R_{{th}}}}{{R_{{2}} + R_{{th}}}} = {R3:.2f}\\,\\Omega$."
    )
)

The value of R3, which is the parallel combination of R2 and Rth, is $R_{3} = \frac{R_{2} \cdot R_{th}}{R_{2} + R_{th}} = 683.12\,\Omega$.

### Results:

In [11]:
display(
    Markdown(
        f"For $V_{{CC}} = {Vcc:.2f}\\,\\text{{V}}$ and $I_{{C, q2}} = {Ic_q2:.2f}\\,\\text{{A}}$ the calculated values are:"
    )
)
display(
    Markdown(
        f"$R_{{4}} = {R4:.2f}\\,\\Omega, R_{{3}} = {R3:.2f}\\,\\Omega, R_{{2}} = {R2:.2f}\\,\\Omega$"
    )
)

For $V_{CC} = 15.54\,\text{V}$ and $I_{C, q2} = 0.30\,\text{A}$ the calculated values are:

$R_{4} = 5.16\,\Omega, R_{3} = 683.12\,\Omega, R_{2} = 3238.94\,\Omega$